In [ ]:
import openeo
from jupytergis import GISDocument
from openeo.processes import process

connection = openeo.connect(url="openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()

In [ ]:
connection.list_collections()

In [ ]:
cube = connection.load_collection(
    "SENTINEL2_L2A",
    bands=["B02", "B03", "B04", "B11"],
    temporal_extent=["2022-04-15", "2022-06-15"],
)

cube = cube / 10000.0

# NDWI = (GREEN - NIR) / (GREEN + NIR)
ndsi = cube.ndvi(nir="2", red="4")

cube = cube.reduce_dimension(
    dimension="t",
    reducer="max",
)

ndsi_vis = (ndsi + 1) / 2

ndsi_png = ndsi_vis.linear_scale_range(
    input_min=0,
    input_max=1,
    output_min=0,
    output_max=255,
)
cube

In [ ]:
result = ndsi_png.save_result(format="PNG")

In [ ]:
doc = GISDocument(latitude=40.75, longitude=-73.95, zoom=12)
await doc.ready()
doc.add_openeo_tile_layer(result)
doc